# Colab: attribute-based watermarking

GPU runtime recommended. Flow: clone → CPRF (Go) → PRC (Rust) → Python packages → Hugging Face login → `app.py` or benchmarks.

Put `HF_TOKEN=…` in `PROJECT_ROOT/.env` (optional) so login is non-interactive after §4 installs `python-dotenv`.

After `git pull`, use **Runtime → Restart session** (or restart the kernel) and run the setup cells again so Python picks up changed repo files.


In [ ]:
import sys

assert sys.version_info >= (3, 11), "Runtime → Change runtime type → Python 3.11+"

import torch

print("Python:", sys.version.split()[0], "| CUDA:", torch.cuda.is_available(), end="")
if torch.cuda.is_available():
    print(" —", torch.cuda.get_device_name(0))
else:
    print()


## 1. Clone

Edit `REPO_OWNER` / `REPO_NAME` for a fork. Tree: `/content/<REPO_NAME>`.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

REPO_OWNER = "maxraffel"
REPO_NAME = "attribute-based-watermarking"
PROJECT_ROOT = Path("/content") / REPO_NAME
url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"

if not (PROJECT_ROOT / "watermarking.py").is_file():
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)
    subprocess.run(["git", "clone", "--depth", "1", url, str(PROJECT_ROOT)], check=True)
else:
    print("Repo present:", PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
WATERMARK_LLM_ID = "meta-llama/Llama-3.2-1B-Instruct"
print("cwd:", PROJECT_ROOT.resolve())


## 2. Build CPRF (`cprf.so`)

Linux shared object. If `go` is missing or older than **1.24**, Go **1.24.4** is installed from [go.dev/dl](https://go.dev/dl) into `/usr/local/go` (requires `curl`).


In [ ]:
import os
import platform
import re
import shutil
import subprocess
from pathlib import Path


def _go_version_line() -> str | None:
    exe = shutil.which("go")
    if not exe:
        return None
    r = subprocess.run([exe, "version"], capture_output=True, text=True, timeout=60)
    if r.returncode != 0:
        return None
    return r.stdout


def _go_tuple_from_version(stdout: str) -> tuple[int, int, int] | None:
    m = re.search(r"go(\d+)\.(\d+)(?:\.(\d+))?", stdout)
    if not m:
        return None
    return int(m.group(1)), int(m.group(2)), int(m.group(3) or 0)


def _go_ok() -> bool:
    line = _go_version_line()
    if not line:
        return False
    t = _go_tuple_from_version(line)
    return t is not None and t >= (1, 24, 0)


def _prepend_path(bin_dir: str) -> None:
    os.environ["PATH"] = bin_dir + os.pathsep + os.environ.get("PATH", "")


def _ensure_go(timeout_s: int = 420) -> str:
    if _go_ok():
        g = shutil.which("go")
        if not g:
            raise RuntimeError("go in PATH reported ok but shutil.which failed")
        return g

    local = Path("/usr/local/go/bin/go")
    if local.is_file():
        _prepend_path(str(local.parent))
        if _go_ok():
            return str(local)

    if not shutil.which("curl"):
        raise RuntimeError("curl is required to install Go on this VM")

    GO_VER = "1.24.4"
    arch = {
        "x86_64": "amd64",
        "AMD64": "amd64",
        "aarch64": "arm64",
        "arm64": "arm64",
    }.get(platform.machine(), "amd64")
    tgz = Path(f"/tmp/go{GO_VER}.linux-{arch}.tar.gz")
    url = f"https://go.dev/dl/go{GO_VER}.linux-{arch}.tar.gz"
    print("Installing Go", GO_VER, "from", url)
    subprocess.run(
        [
            "curl",
            "-fL",
            "--retry",
            "3",
            "--connect-timeout",
            "30",
            "-o",
            str(tgz),
            url,
        ],
        check=True,
        stdin=subprocess.DEVNULL,
        timeout=timeout_s,
    )
    if not tgz.is_file() or tgz.stat().st_size < 1024 * 1024:
        raise RuntimeError("Go download failed or file too small")

    goroot = Path("/usr/local/go")
    if goroot.is_dir():
        shutil.rmtree(goroot)
    subprocess.run(
        ["tar", "-C", "/usr/local", "-xzf", str(tgz)],
        check=True,
        stdin=subprocess.DEVNULL,
        timeout=timeout_s,
    )
    _prepend_path("/usr/local/go/bin")
    g = "/usr/local/go/bin/go"
    subprocess.run([g, "version"], check=True, timeout=60)
    return g


cprf_dir = PROJECT_ROOT / "cprf"
so_path = cprf_dir / "cprf.so"
go_exe = _ensure_go()

if not so_path.is_file():
    subprocess.run(
        [go_exe, "build", "-o", "cprf.so", "-buildmode=c-shared", "cprf.go"],
        cwd=cprf_dir,
        check=True,
    )
assert so_path.is_file(), "cprf.so missing"
print("CPRF:", so_path)


## 3. Build PRC (Rust + maturin)

Installs Rust via rustup if needed, then `maturin build --release` and `pip install` the newest wheel under `prc/target/wheels/`.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

cargo_bin = Path.home() / ".cargo" / "bin"
if not (cargo_bin / "rustc").exists():
    subprocess.run(
        "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y",
        shell=True,
        check=True,
    )
os.environ["PATH"] = str(cargo_bin) + os.pathsep + os.environ.get("PATH", "")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "maturin"], check=True)

cp = subprocess.run(
    ["maturin", "build", "--release", "-m", "prc/Cargo.toml"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)
if cp.returncode != 0:
    if cp.stdout:
        print(cp.stdout)
    if cp.stderr:
        print(cp.stderr, file=sys.stderr)
    cp.check_returncode()

wheel_dir = PROJECT_ROOT / "prc" / "target" / "wheels"
wheels = sorted(wheel_dir.glob("*.whl"), key=lambda p: p.stat().st_mtime, reverse=True)
if not wheels:
    raise FileNotFoundError("No wheel in prc/target/wheels after maturin build")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(wheels[0])], check=True)
print("PRC:", wheels[0].name)


## 4. Python dependencies

Colab ships PyTorch with CUDA. `sympy` is reinstalled first (avoids a known Colab/transformers import clash). Adds packages used by optional chart/matrix cells.


In [ ]:
%pip install -q --upgrade --force-reinstall "sympy>=1.13,<2"
%pip install -q "transformers==5.5.4" "rich>=15" "keybert>=0.9" sentencepiece accelerate python-dotenv matplotlib pandas


## 5. Hugging Face login

Accept the **Llama 3.2** license on the model card. With `HF_TOKEN` / `HUGGING_FACE_HUB_TOKEN` in the environment or in `PROJECT_ROOT/.env`, login is automatic; otherwise this cell runs `notebook_login()`.


In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from huggingface_hub import login, notebook_login

_env = PROJECT_ROOT / ".env"
if _env.is_file():
    load_dotenv(_env)

_token = os.environ.get("HF_TOKEN", "").strip() or os.environ.get(
    "HUGGING_FACE_HUB_TOKEN", ""
).strip()
if _token:
    login(token=_token, add_to_git_credential=False)
    print("HF: logged in from token.")
else:
    notebook_login()


## 6. `git pull`

Optional. Re-run §2–§3 only if native code changed.


In [ ]:
import subprocess

if not (PROJECT_ROOT / ".git").is_dir():
    raise FileNotFoundError("Run §1 first.")

subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
log = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "log", "-1", "--oneline"],
    capture_output=True,
    text=True,
    check=True,
)
print(log.stdout.strip())


## 7. Demo (`app.py`)

Edit `APP_*` / `WATERMARK_LLM_ID` in the next cell. `SystemExit` from failed checks is caught so Jupyter does not print a traceback for a nonzero exit code.


In [ ]:
import os
import runpy

os.chdir(PROJECT_ROOT)

APP_CODE_LENGTH = 300
APP_WM_BIT_REDUNDANCY = 1
os.environ["APP_CODE_LENGTH"] = str(int(APP_CODE_LENGTH))
os.environ["APP_WM_BIT_REDUNDANCY"] = str(int(APP_WM_BIT_REDUNDANCY))

WATERMARK_LLM_ID = "meta-llama/Llama-3.2-1B-Instruct"

import watermarking as wm

wm.set_llm_model_id(WATERMARK_LLM_ID)

try:
    runpy.run_path(str(PROJECT_ROOT / "app.py"), run_name="__main__")
except SystemExit as exc:
    if exc.code not in (0, None):
        print("app.py exit:", repr(exc.code), "(0 = all checks passed)")


## 8. Policy benchmark

Edit `BENCHMARK_*` in the next cell. Prompts: list of `"id:prompt"` (split on first `:`); empty list uses `DEFAULT_PROMPT_CASES`. `BENCHMARK_LLM_MODEL=None` uses `WATERMARK_LLM_ID` from §1.


In [ ]:
import os
import sys
from pathlib import Path

BENCHMARK_RUNS = 10
BENCHMARK_CODE_LENGTH = 300
BENCHMARK_WM_BIT_REDUNDANCY = 1
BENCHMARK_MODULUS = 1024
BENCHMARK_REUSE_KEY = False
BENCHMARK_LLM_MODEL: str | None = None
BENCHMARK_FORCE_PLAIN_TABLE = False
BENCHMARK_PROMPT_CASES: list[str] = [
    (
    "medicine_stem_cell:"
    "Explain how stem cell therapy is being used in regenerative medicine."
),
(
"economics_min_wage:"
"Explain the economic effects of raising the minimum wage on employment and businesses."
),
    (
    "art_surrealism:"
    "Explain how Surrealist artists used dream imagery to challenge reality and logic."
),
(
"software_breakthroughs:"
"Break down the most influential software breakthroughs in history."
),
(
"sports_strategy_performance:"
"Explain the role of strategy and teamwork in achieving success in sports."
),
]

os.chdir(PROJECT_ROOT)
root = Path(PROJECT_ROOT)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

if BENCHMARK_FORCE_PLAIN_TABLE:
    os.environ["BENCHMARK_PLAIN_TABLE"] = "1"

import watermarking as wm
from benchmark_policy_detection import (
    DEFAULT_PROMPT_CASES,
    make_benchmark_console,
    parse_prompt_case,
    run_benchmark,
)

llm = BENCHMARK_LLM_MODEL
if (llm is None or not str(llm).strip()) and "WATERMARK_LLM_ID" in globals():
    llm = WATERMARK_LLM_ID
if llm and str(llm).strip():
    wm.set_llm_model_id(str(llm).strip())

cases = (
    [parse_prompt_case(s) for s in BENCHMARK_PROMPT_CASES]
    if BENCHMARK_PROMPT_CASES
    else list(DEFAULT_PROMPT_CASES)
)

exit_code = run_benchmark(
    prompt_cases=cases,
    runs=int(BENCHMARK_RUNS),
    modulus=int(BENCHMARK_MODULUS),
    code_length=int(BENCHMARK_CODE_LENGTH),
    fresh_key_per_trial=not BENCHMARK_REUSE_KEY,
    console=make_benchmark_console(),
    llm_model_id=str(llm).strip() if llm and str(llm).strip() else None,
    wm_bit_redundancy=int(BENCHMARK_WM_BIT_REDUNDANCY),
)
print("run_benchmark exit:", exit_code, "(0 = strict pass on every run)")


## 9. Chart: FPR vs code length

One benchmark run per length; also plots PRC random baseline. Writes `benchmark_fpr_vs_code_length.{json,png}` (and `_ci.png`) under the repo root.


In [ ]:
import json
import os
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt

os.chdir(PROJECT_ROOT)
root = Path(PROJECT_ROOT)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import watermarking as wm
from benchmark_policy_detection import (
    DEFAULT_PROMPT_CASES,
    make_benchmark_console,
    micro_fpr_wilson,
    parse_prompt_case,
    prc_random_detect_positive_rate,
    run_benchmark_with_summary,
    wilson_score_interval,
)

FPR_CHART_CODE_LENGTHS = [128, 256, 384]
FPR_CHART_RUNS = 3
FPR_CHART_MODULUS = 1024
FPR_CHART_WM_BIT_REDUNDANCY = 1
FPR_CHART_REUSE_KEY = False
FPR_CHART_PRC_MONTE_CARLO = 100_000
FPR_CHART_RNG = random.Random(42)
FPR_CHART_PROMPT_CASES: list[str] = [
    (
    "medicine_stem_cell:"
    "Explain how stem cell therapy is being used in regenerative medicine."
),
(
"economics_min_wage:"
"Explain the economic effects of raising the minimum wage on employment and businesses."
),
    (
    "art_surrealism:"
    "Explain how Surrealist artists used dream imagery to challenge reality and logic."
),
(
"software_breakthroughs:"
"Break down the most influential software breakthroughs in history."
),
(
"sports_strategy_performance:"
"Explain the role of strategy and teamwork in achieving success in sports."
),
]
FPR_CHART_LLM_MODEL: str | None = None

llm = FPR_CHART_LLM_MODEL
if (llm is None or not str(llm).strip()) and "WATERMARK_LLM_ID" in globals():
    llm = WATERMARK_LLM_ID
if llm and str(llm).strip():
    wm.set_llm_model_id(str(llm).strip())

cases = (
    [parse_prompt_case(s) for s in FPR_CHART_PROMPT_CASES]
    if FPR_CHART_PROMPT_CASES
    else list(DEFAULT_PROMPT_CASES)
)
console = make_benchmark_console()

lengths: list[int] = []
scheme_fpr_all: list[float] = []
scheme_fpr_xmatch: list[float] = []
scheme_fpr_all_lo: list[float] = []
scheme_fpr_all_hi: list[float] = []
scheme_fpr_xmatch_lo: list[float] = []
scheme_fpr_xmatch_hi: list[float] = []
prc_random_fpr: list[float] = []
prc_random_fpr_lo: list[float] = []
prc_random_fpr_hi: list[float] = []
exit_codes: list[int] = []

for L in FPR_CHART_CODE_LENGTHS:
    print(f"--- code_length={L}: PRC random ({FPR_CHART_PRC_MONTE_CARLO} trials) ---")
    r_rand, fp_mc = prc_random_detect_positive_rate(
        int(L), int(FPR_CHART_PRC_MONTE_CARLO), rng=FPR_CHART_RNG
    )
    prc_random_fpr.append(float(r_rand))
    pl, ph = wilson_score_interval(fp_mc, int(FPR_CHART_PRC_MONTE_CARLO))
    prc_random_fpr_lo.append(pl)
    prc_random_fpr_hi.append(ph)

    print(f"--- code_length={L}: benchmark ({FPR_CHART_RUNS} runs/prompt) ---")
    ex, summary = run_benchmark_with_summary(
        prompt_cases=cases,
        runs=int(FPR_CHART_RUNS),
        modulus=int(FPR_CHART_MODULUS),
        code_length=int(L),
        fresh_key_per_trial=not FPR_CHART_REUSE_KEY,
        console=console,
        llm_model_id=str(llm).strip() if llm and str(llm).strip() else None,
        wm_bit_redundancy=int(FPR_CHART_WM_BIT_REDUNDANCY),
        quiet=True,
    )
    lengths.append(int(L))
    exit_codes.append(int(ex))

    fa, fa_lo, fa_hi = micro_fpr_wilson(summary.roll, summary.prompt_cases)
    fx, fx_lo, fx_hi = micro_fpr_wilson(summary.roll_xmatch, summary.prompt_cases)
    scheme_fpr_all.append(float(fa) if fa >= 0.0 else float("nan"))
    scheme_fpr_xmatch.append(float(fx) if fx >= 0.0 else float("nan"))
    scheme_fpr_all_lo.append(fa_lo if fa >= 0.0 else float("nan"))
    scheme_fpr_all_hi.append(fa_hi if fa >= 0.0 else float("nan"))
    scheme_fpr_xmatch_lo.append(fx_lo if fx >= 0.0 else float("nan"))
    scheme_fpr_xmatch_hi.append(fx_hi if fx >= 0.0 else float("nan"))

    print(
        f"code_length={L} exit={ex} FPR_all={scheme_fpr_all[-1]:.6g} "
        f"FPR_xmatch={scheme_fpr_xmatch[-1]:.6g} PRC_random={prc_random_fpr[-1]:.6g}"
    )

json_path = root / "benchmark_fpr_vs_code_length.json"
png_path = root / "benchmark_fpr_vs_code_length.png"
png_ci = root / "benchmark_fpr_vs_code_length_ci.png"

payload = {
    "code_lengths": lengths,
    "scheme_fpr_all_runs": scheme_fpr_all,
    "scheme_fpr_x_matched_runs_only": scheme_fpr_xmatch,
    "prc_random_detect_rate": prc_random_fpr,
    "scheme_fpr_all_ci_low": scheme_fpr_all_lo,
    "scheme_fpr_all_ci_high": scheme_fpr_all_hi,
    "scheme_fpr_x_matched_ci_low": scheme_fpr_xmatch_lo,
    "scheme_fpr_x_matched_ci_high": scheme_fpr_xmatch_hi,
    "prc_random_detect_rate_ci_low": prc_random_fpr_lo,
    "prc_random_detect_rate_ci_high": prc_random_fpr_hi,
    "benchmark_exit_codes": exit_codes,
    "runs_per_prompt": int(FPR_CHART_RUNS),
    "prc_monte_carlo_trials": int(FPR_CHART_PRC_MONTE_CARLO),
    "modulus": int(FPR_CHART_MODULUS),
    "wm_bit_redundancy": int(FPR_CHART_WM_BIT_REDUNDANCY),
}
json_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("Wrote", json_path)


def _asym_errbar(y_vals, lo, hi):
    el, eh = [], []
    for y, a, b in zip(y_vals, lo, hi):
        if y != y or a != a or b != b:
            el.append(float("nan"))
            eh.append(float("nan"))
        else:
            el.append(y - a)
            eh.append(b - y)
    return [el, eh]


fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(lengths, scheme_fpr_all, "o-", label="Scheme FPR (all runs)")
ax.plot(lengths, scheme_fpr_xmatch, "s--", label="Scheme FPR (x matched)")
ax.plot(lengths, prc_random_fpr, "^-", label="PRC baseline")
ax.set_xlabel("Code length (tokens)")
ax.set_ylabel("False positive rate")
ax.grid(True, alpha=0.3)
ax.legend(loc="best")
fig.tight_layout()
fig.savefig(png_path, dpi=150)
plt.show()
print("Wrote", png_path)

fig2, ax2 = plt.subplots(figsize=(8, 5))
ax2.errorbar(
    lengths,
    scheme_fpr_all,
    yerr=_asym_errbar(scheme_fpr_all, scheme_fpr_all_lo, scheme_fpr_all_hi),
    fmt="o-",
    capsize=4,
    label="Scheme FPR (all runs)",
)
ax2.errorbar(
    lengths,
    scheme_fpr_xmatch,
    yerr=_asym_errbar(scheme_fpr_xmatch, scheme_fpr_xmatch_lo, scheme_fpr_xmatch_hi),
    fmt="s--",
    capsize=4,
    label="Scheme FPR (x matched)",
)
ax2.errorbar(
    lengths,
    prc_random_fpr,
    yerr=_asym_errbar(prc_random_fpr, prc_random_fpr_lo, prc_random_fpr_hi),
    fmt="^-",
    capsize=4,
    label="PRC baseline",
)
ax2.set_xlabel("Code length (tokens)")
ax2.set_ylabel("False positive rate")
ax2.grid(True, alpha=0.3)
ax2.legend(loc="best")
fig2.tight_layout()
fig2.savefig(png_ci, dpi=150)
plt.show()
print("Wrote", png_ci)


## 10. Chart: TPR vs WM bit redundancy

Fixed logical code length; sweeps redundancy. Exports `benchmark_tpr_vs_wm_bit_redundancy.{json,png,_ci.png}`.


In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt

os.chdir(PROJECT_ROOT)
root = Path(PROJECT_ROOT)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import watermarking as wm
from benchmark_policy_detection import (
    DEFAULT_PROMPT_CASES,
    make_benchmark_console,
    micro_tpr_wilson,
    parse_prompt_case,
    run_benchmark_with_summary,
)

TPR_CHART_WM_BIT_REDUNDANCY_VALUES = [1, 3, 5]
TPR_CHART_CODE_LENGTH = 100
TPR_CHART_RUNS = 3
TPR_CHART_MODULUS = 1024
TPR_CHART_REUSE_KEY = False
TPR_CHART_PROMPT_CASES: list[str] = [
(
    "medicine_stem_cell:"
    "Explain how stem cell therapy is being used in regenerative medicine."
),
(
"economics_min_wage:"
"Explain the economic effects of raising the minimum wage on employment and businesses."
),
    (
    "art_surrealism:"
    "Explain how Surrealist artists used dream imagery to challenge reality and logic."
),
(
"software_breakthroughs:"
"Break down the most influential software breakthroughs in history."
),
(
"sports_strategy_performance:"
"Explain the role of strategy and teamwork in achieving success in sports."
),
]
TPR_CHART_LLM_MODEL: str | None = None

llm = TPR_CHART_LLM_MODEL
if (llm is None or not str(llm).strip()) and "WATERMARK_LLM_ID" in globals():
    llm = WATERMARK_LLM_ID
if llm and str(llm).strip():
    wm.set_llm_model_id(str(llm).strip())

cases = (
    [parse_prompt_case(s) for s in TPR_CHART_PROMPT_CASES]
    if TPR_CHART_PROMPT_CASES
    else list(DEFAULT_PROMPT_CASES)
)
console = make_benchmark_console()

redundancies: list[int] = []
tpr_all: list[float] = []
tpr_xmatch: list[float] = []
tpr_all_lo: list[float] = []
tpr_all_hi: list[float] = []
tpr_xmatch_lo: list[float] = []
tpr_xmatch_hi: list[float] = []
exit_codes: list[int] = []

for R in TPR_CHART_WM_BIT_REDUNDANCY_VALUES:
    print(f"--- wm_bit_redundancy={R} code_length={TPR_CHART_CODE_LENGTH} ---")
    ex, summary = run_benchmark_with_summary(
        prompt_cases=cases,
        runs=int(TPR_CHART_RUNS),
        modulus=int(TPR_CHART_MODULUS),
        code_length=int(TPR_CHART_CODE_LENGTH),
        fresh_key_per_trial=not TPR_CHART_REUSE_KEY,
        console=console,
        llm_model_id=str(llm).strip() if llm and str(llm).strip() else None,
        wm_bit_redundancy=int(R),
        quiet=True,
    )
    redundancies.append(int(R))
    exit_codes.append(int(ex))

    ta, ta_lo, ta_hi = micro_tpr_wilson(summary.roll, summary.prompt_cases)
    tx, tx_lo, tx_hi = micro_tpr_wilson(summary.roll_xmatch, summary.prompt_cases)
    tpr_all.append(float(ta) if ta >= 0.0 else float("nan"))
    tpr_xmatch.append(float(tx) if tx >= 0.0 else float("nan"))
    tpr_all_lo.append(ta_lo if ta >= 0.0 else float("nan"))
    tpr_all_hi.append(ta_hi if ta >= 0.0 else float("nan"))
    tpr_xmatch_lo.append(tx_lo if tx >= 0.0 else float("nan"))
    tpr_xmatch_hi.append(tx_hi if tx >= 0.0 else float("nan"))

    print(
        f"wm_bit_redundancy={R} exit={ex} TPR_all={tpr_all[-1]:.6g} TPR_xmatch={tpr_xmatch[-1]:.6g}"
    )

json_path = root / "benchmark_tpr_vs_wm_bit_redundancy.json"
png_path = root / "benchmark_tpr_vs_wm_bit_redundancy.png"
png_ci = root / "benchmark_tpr_vs_wm_bit_redundancy_ci.png"

output_lens = [int(TPR_CHART_CODE_LENGTH) * int(R) for R in redundancies]
payload = {
    "wm_bit_redundancy": redundancies,
    "tpr_all_runs": tpr_all,
    "tpr_x_matched_runs_only": tpr_xmatch,
    "tpr_all_runs_ci_low": tpr_all_lo,
    "tpr_all_runs_ci_high": tpr_all_hi,
    "tpr_x_matched_ci_low": tpr_xmatch_lo,
    "tpr_x_matched_ci_high": tpr_xmatch_hi,
    "benchmark_exit_codes": exit_codes,
    "code_length": int(TPR_CHART_CODE_LENGTH),
    "runs_per_prompt": int(TPR_CHART_RUNS),
    "modulus": int(TPR_CHART_MODULUS),
}
json_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("Wrote", json_path)


def _asym_errbar(y_vals, lo, hi):
    el, eh = [], []
    for y, a, b in zip(y_vals, lo, hi):
        if y != y or a != a or b != b:
            el.append(float("nan"))
            eh.append(float("nan"))
        else:
            el.append(y - a)
            eh.append(b - y)
    return [el, eh]


fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(redundancies, tpr_all, "o-", label="All runs")
ax.plot(redundancies, tpr_xmatch, "s--", label="x matched runs")
ax.axhline(1.0, color="gray", linestyle=":", linewidth=1.0)
ax.set_xticks(redundancies)
ax.set_xticklabels([f"{R}\n({L} tokens)" for R, L in zip(redundancies, output_lens)])
ax.set_xlabel("Redundancy / output length")
ax.set_ylabel("True positive rate")
ax.set_ylim(-0.02, 1.02)
ax.grid(True, alpha=0.3)
ax.legend(loc="best")
fig.tight_layout()
fig.savefig(png_path, dpi=150)
plt.show()
print("Wrote", png_path)

fig2, ax2 = plt.subplots(figsize=(8, 5))
ax2.errorbar(
    redundancies,
    tpr_all,
    yerr=_asym_errbar(tpr_all, tpr_all_lo, tpr_all_hi),
    fmt="o-",
    capsize=4,
    label="All runs",
)
ax2.errorbar(
    redundancies,
    tpr_xmatch,
    yerr=_asym_errbar(tpr_xmatch, tpr_xmatch_lo, tpr_xmatch_hi),
    fmt="s--",
    capsize=4,
    label="x matched runs",
)
ax2.axhline(1.0, color="gray", linestyle=":", linewidth=1.0)
ax2.set_xticks(redundancies)
ax2.set_xticklabels([f"{R}\n({L} tokens)" for R, L in zip(redundancies, output_lens)])
ax2.set_xlabel("Redundancy / output length")
ax2.set_ylabel("True positive rate")
ax2.set_ylim(-0.02, 1.02)
ax2.grid(True, alpha=0.3)
ax2.legend(loc="best")
fig2.tight_layout()
fig2.savefig(png_ci, dpi=150)
plt.show()
print("Wrote", png_ci)


## 11. Matrix: detection vs attributed label

`|V| × |V|` conditional rates; CSV/JSON under `benchmark_label_conditioned_detection_matrix*`.


In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd

os.chdir(PROJECT_ROOT)
root = Path(PROJECT_ROOT)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import watermarking as wm
from benchmark_policy_detection import (
    DEFAULT_PROMPT_CASES,
    make_benchmark_console,
    parse_prompt_case,
    run_benchmark_label_conditioned_matrix,
)

MATRIX_RUNS = 3
MATRIX_CODE_LENGTH = 300
MATRIX_WM_BIT_REDUNDANCY = 1
MATRIX_MODULUS = 1024
MATRIX_REUSE_KEY = False
MATRIX_PROMPT_CASES: list[str] = [
(
    "medicine_stem_cell:"
    "Explain how stem cell therapy is being used in regenerative medicine."
),
(
"economics_min_wage:"
"Explain the economic effects of raising the minimum wage on employment and businesses."
),
    (
    "art_surrealism:"
    "Explain how Surrealist artists used dream imagery to challenge reality and logic."
),
(
"software_breakthroughs:"
"Break down the most influential software breakthroughs in history."
),
(
"sports_strategy_performance:"
"Explain the role of strategy and teamwork in achieving success in sports."
),
]
MATRIX_LLM_MODEL: str | None = None

llm = MATRIX_LLM_MODEL
if (llm is None or not str(llm).strip()) and "WATERMARK_LLM_ID" in globals():
    llm = WATERMARK_LLM_ID
if llm and str(llm).strip():
    wm.set_llm_model_id(str(llm).strip())

cases = (
    [parse_prompt_case(s) for s in MATRIX_PROMPT_CASES]
    if MATRIX_PROMPT_CASES
    else list(DEFAULT_PROMPT_CASES)
)

ex, mx = run_benchmark_label_conditioned_matrix(
    prompt_cases=cases,
    runs=int(MATRIX_RUNS),
    modulus=int(MATRIX_MODULUS),
    code_length=int(MATRIX_CODE_LENGTH),
    fresh_key_per_trial=not MATRIX_REUSE_KEY,
    console=make_benchmark_console(),
    llm_model_id=str(llm).strip() if llm and str(llm).strip() else None,
    wm_bit_redundancy=int(MATRIX_WM_BIT_REDUNDANCY),
    quiet=True,
)

vocab = list(mx.vocab)
num_df = pd.DataFrame(mx.numerators).T.loc[vocab, vocab]
den_df = pd.DataFrame(mx.denominators).T.loc[vocab, vocab]
pct_df = (num_df / den_df.replace(0, pd.NA)) * 100.0
frac_df = num_df.astype(str) + "/" + den_df.astype(str)

print(f"exit={ex} runs/prompt={MATRIX_RUNS} |V|={len(vocab)}")
display(pct_df.round(2))
display(frac_df)

prefix = root / "benchmark_label_conditioned_detection_matrix"
pct_df.to_csv(prefix.with_name(prefix.name + "_percent.csv"), encoding="utf-8")
frac_df.to_csv(prefix.with_name(prefix.name + "_fraction.csv"), encoding="utf-8")
num_df.to_csv(prefix.with_name(prefix.name + "_numerator.csv"), encoding="utf-8")
den_df.to_csv(prefix.with_name(prefix.name + "_denominator.csv"), encoding="utf-8")
payload = {
    "vocab": vocab,
    "numerators": mx.numerators,
    "denominators": mx.denominators,
    "rates": mx.rates,
    "matrix_exit_code": int(ex),
    "strict_protocol_ok": bool(mx.strict_protocol_ok),
    "runs_per_prompt": int(mx.runs_per_prompt),
    "code_length": int(mx.code_length),
    "wm_bit_redundancy": int(mx.wm_bit_redundancy),
    "modulus": int(mx.modulus),
    "prompt_case_ids": [sid for sid, _ in mx.prompt_cases],
}
jpath = prefix.with_name(prefix.name + ".json")
jpath.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("Wrote", jpath.name, "and CSVs under", root)


## 12. Matrix: detection vs prompt id

Same as §11 but columns are prompt ids; includes x-matched subset. Files `benchmark_prompt_conditioned_detection_matrix*`.


In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd

os.chdir(PROJECT_ROOT)
root = Path(PROJECT_ROOT)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import watermarking as wm
from benchmark_policy_detection import (
    DEFAULT_PROMPT_CASES,
    make_benchmark_console,
    parse_prompt_case,
    run_benchmark_prompt_conditioned_matrix,
)

PROMPT_MATRIX_RUNS = 3
PROMPT_MATRIX_CODE_LENGTH = 300
PROMPT_MATRIX_WM_BIT_REDUNDANCY = 1
PROMPT_MATRIX_MODULUS = 1024
PROMPT_MATRIX_REUSE_KEY = False
PROMPT_MATRIX_PROMPT_CASES: list[str] = [
    "sports_econ:Explain the economic nuance and impact of Drake Maye during his college football career at North Carolina.",
    "art_software:Explain how software has transformed the art world.",
    "medicine_software:Explain how software has transformed the practice of medicine."
]
PROMPT_MATRIX_LLM_MODEL: str | None = None

llm = PROMPT_MATRIX_LLM_MODEL
if (llm is None or not str(llm).strip()) and "WATERMARK_LLM_ID" in globals():
    llm = WATERMARK_LLM_ID
if llm and str(llm).strip():
    wm.set_llm_model_id(str(llm).strip())

cases = (
    [parse_prompt_case(s) for s in PROMPT_MATRIX_PROMPT_CASES]
    if PROMPT_MATRIX_PROMPT_CASES
    else list(DEFAULT_PROMPT_CASES)
)

ex_pm, pmx = run_benchmark_prompt_conditioned_matrix(
    prompt_cases=cases,
    runs=int(PROMPT_MATRIX_RUNS),
    modulus=int(PROMPT_MATRIX_MODULUS),
    code_length=int(PROMPT_MATRIX_CODE_LENGTH),
    fresh_key_per_trial=not PROMPT_MATRIX_REUSE_KEY,
    console=make_benchmark_console(),
    llm_model_id=str(llm).strip() if llm and str(llm).strip() else None,
    wm_bit_redundancy=int(PROMPT_MATRIX_WM_BIT_REDUNDANCY),
    quiet=True,
)

vocab = list(pmx.vocab)
cols = list(pmx.column_prompt_ids)
num_df = pd.DataFrame(pmx.numerators).T.loc[vocab, cols]
den_df = pd.DataFrame(pmx.denominators).T.loc[vocab, cols]
pct_df = (num_df / den_df.replace(0, pd.NA)) * 100.0
frac_df = num_df.astype(str) + "/" + den_df.astype(str)

num_xm = pd.DataFrame(pmx.numerators_xmatch).T.loc[vocab, cols]
den_xm = pd.DataFrame(pmx.denominators_xmatch).T.loc[vocab, cols]
pct_xm = (num_xm / den_xm.replace(0, pd.NA)) * 100.0
frac_xm = num_xm.astype(str) + "/" + den_xm.astype(str)

print(f"exit={ex_pm} runs/prompt={PROMPT_MATRIX_RUNS} |V|={len(vocab)} |P|={len(cols)}")
display(pct_df.round(2))
display(pct_xm.round(2))

prefix = root / "benchmark_prompt_conditioned_detection_matrix"
prefix_xm = root / "benchmark_prompt_conditioned_detection_matrix_xmatch"
for df, p, suffix in (
    (pct_df, prefix, "_percent.csv"),
    (frac_df, prefix, "_fraction.csv"),
    (num_df, prefix, "_numerator.csv"),
    (den_df, prefix, "_denominator.csv"),
    (pct_xm, prefix_xm, "_percent.csv"),
    (frac_xm, prefix_xm, "_fraction.csv"),
    (num_xm, prefix_xm, "_numerator.csv"),
    (den_xm, prefix_xm, "_denominator.csv"),
):
    df.to_csv(p.with_name(p.name + suffix), encoding="utf-8")

json_payload = {
    "vocab": vocab,
    "cols_prompt_id": cols,
    "numerators": pmx.numerators,
    "denominators": pmx.denominators,
    "rates": pmx.rates,
    "numerators_x_matched_runs_only": pmx.numerators_xmatch,
    "denominators_x_matched_runs_only": pmx.denominators_xmatch,
    "rates_x_matched_runs_only": pmx.rates_xmatch,
    "matrix_exit_code": int(ex_pm),
    "strict_protocol_ok": bool(pmx.strict_protocol_ok),
    "runs_per_prompt": int(pmx.runs_per_prompt),
    "code_length": int(pmx.code_length),
    "wm_bit_redundancy": int(pmx.wm_bit_redundancy),
    "modulus": int(pmx.modulus),
    "prompt_case_ids": [sid for sid, _ in pmx.prompt_cases],
}
prefix.with_name(prefix.name + ".json").write_text(
    json.dumps(json_payload, indent=2), encoding="utf-8"
)
print("Wrote matrices under", root)


## 13. Custom prompt (`app.main()`)

Reloads `app`, patches `PROMPT` / lengths, runs `main()` without editing `app.py`.


In [ ]:
import importlib
import os
import sys
from pathlib import Path

os.chdir(PROJECT_ROOT)
root = Path(PROJECT_ROOT)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import watermarking as wm
import app as app_mod

APP_CUSTOM_PROMPT = "Explain how advances in battery chemistry changed electric vehicle adoption."
APP_CUSTOM_CODE_LENGTH = 100
APP_CUSTOM_WM_BIT_REDUNDANCY = 3
APP_CUSTOM_MODULUS = 1024
APP_CUSTOM_LLM_MODEL: str | None = None

llm = APP_CUSTOM_LLM_MODEL
if (llm is None or not str(llm).strip()) and "WATERMARK_LLM_ID" in globals():
    llm = WATERMARK_LLM_ID
if llm and str(llm).strip():
    wm.set_llm_model_id(str(llm).strip())

app_mod = importlib.reload(app_mod)
app_mod.PROMPT = str(APP_CUSTOM_PROMPT)
app_mod.CODE_LENGTH = int(APP_CUSTOM_CODE_LENGTH)
app_mod.WM_BIT_REDUNDANCY = int(APP_CUSTOM_WM_BIT_REDUNDANCY)
app_mod.MODULUS = int(APP_CUSTOM_MODULUS)
app_mod.LLM_MODEL_ID = str(llm).strip() if llm and str(llm).strip() else None

print("app.main()", app_mod.PROMPT[:80], "...")
code = app_mod.main()
print("exit:", code, "(0 = all checks passed)")
